# InnoBERT user manual

This notebook shows installation, CPU/GPU loading, aligned batch inputs, noun-chunk, sentence, and paragraph processing, threshold changes, DataFrame input, output checks, and common errors. Replace the model identifier with the final Hugging Face repository or an extracted local model directory.

## 1. Install

From a cloned repository, install the package and notebook dependencies. Noun-chunk mode also requires the spaCy model.

In [ ]:
%pip install -e ".[noun-chunks,notebook]"
!python -m spacy download en_core_web_lg

## 2. Inspect the available hardware

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Load InnoBERT

`device="auto"` selects CUDA, then Apple MPS, then CPU. You can force `cpu`, `cuda`, or `cuda:0`. A local path must point to the extracted directory, not the ZIP file.

In [ ]:
from innobert import InnoBERT

MODEL_ID = "mustafahci/InnoBERT"  # or: ./model_assets/innobert-final-v1
classifier = InnoBERT.from_pretrained(MODEL_ID, device="auto", spacy_model="en_core_web_lg")

## 4. Already-extracted terms with row-specific contexts

The three lists must align row by row. A scalar industry or year is broadcast to every text. Term mode uses the exact training prompt and the uncategorized gatekeeper by default.

In [ ]:
term_results = classifier.predict(
    ["cloud-based document platform", "automated production system"],
    industry=["Software", "Manufacturing"],
    year=[2024, 2023],
    unit="term",
    include_model_input=True,
)
term_results[["processed_text", "industry", "year", "predicted_labels", "dominant_probability"]]

## 5. Noun-chunk extraction

Each source document expands to zero or more extracted terms. `source_id` and `unit_index` preserve the lineage. The optional filer name removes company self-references from candidate terms.

In [ ]:
sample_text = (
    "Example Corporation introduced a cloud-based analytics platform and an "
    "automated inventory-planning system for retail customers."
)

noun_results = classifier.predict(
    sample_text,
    industry="Software",
    year=2024,
    filer_name="Example Corporation",
    unit="noun_chunk",
)
noun_results[["unit_id", "processed_text", "predicted_labels"]]

## 6. Sentence processing

Sentence mode reproduces the conference-call splitter. It uses raw sentences and the fallback uncategorized rule by default. Industry and year can still be retained as output metadata.

In [ ]:
sentence_results = classifier.predict(
    sample_text,
    industry="Software",
    year=2024,
    unit="sentence",
)
sentence_results[["processed_text", "predicted_labels", "token_count"]]

## 7. Paragraph processing and token windows

Paragraphs are separated at blank lines. Long paragraphs are split into overlapping token windows; the returned probability for each category is the maximum across windows. Inspect `window_count` and `truncated_or_windowed`.

In [ ]:
paragraph_text = """We introduced a cloud-based analytics platform for customers.

The redesigned subscription model automates ordering and inventory planning."""

paragraph_results = classifier.predict(
    paragraph_text,
    unit="paragraph",
    max_length=160,
    long_text_strategy="window",
    stride=32,
)
paragraph_results[["processed_text", "window_count", "predicted_labels"]]

## 8. DataFrame batches

Column names are configurable. For sentence mode below, industry and year remain metadata. Set `context_mode="industry_year"` only if you intentionally want to wrap sentences in the original term-training prompt.

In [ ]:
import pandas as pd

documents = pd.DataFrame({
    "document_id": ["A", "B"],
    "business_text": ["We launched a digital service.", "We automated quality control."],
    "industry_name": ["Software", "Manufacturing"],
    "fyear": [2024, 2023],
})

batch_results = classifier.predict(
    documents,
    unit="sentence",
    text_col="business_text",
    industry_col="industry_name",
    year_col="fyear",
    source_id_col="document_id",
)
batch_results

## 9. Adjust selected thresholds

Unspecified thresholds retain the defaults. Canonical names or aliases such as `product`, `business_model`, and `AI` are accepted.
`dominant_label` is selected only from `predicted_labels`; the eight `prob_*` columns retain all raw model scores.

In [ ]:
adjusted_results = classifier.predict(
    sample_text,
    unit="sentence",
    thresholds={"product": 0.70, "AI": 0.40},
)
adjusted_results[["processed_text", "predicted_labels"]]

## 10. Catch input errors

The example below intentionally supplies two texts but only one industry. The error reports both counts. Other explicit checks cover missing context, invalid years and thresholds, unavailable devices, missing spaCy models, unsupported units, and long inputs.

In [ ]:
try:
    classifier.predict(
        ["first term", "second term"],
        industry=["Software"],
        year=[2024, 2023],
        unit="term",
    )
except (TypeError, ValueError, RuntimeError, ImportError, OSError) as error:
    print(type(error).__name__ + ":", error)

## 11. Save results

CSV is universal but stores `predicted_labels` as text. Parquet preserves list-like columns more naturally.

In [ ]:
batch_results.to_csv("innobert_results.csv", index=False)
batch_results.to_parquet("innobert_results.parquet", index=False)